# Paso 1 · Carga y verificación del dataset (Lending Club)

**Objetivo de este notebook:** comprobar que Colab ve tu archivo, que se carga *completo* (sin muestreo, como exige el enunciado) y entender qué trae la columna `loan_status`, que es la base de la variable objetivo `default`.

**Por qué empezamos aquí:** antes de modelar hay que saber exactamente con cuántas filas trabajamos y qué categorías tiene el target. De eso dependen todas las decisiones posteriores.

Ejecuta las celdas en orden (Shift + Enter) y copia lo que salga en las celdas marcadas con 👉.

## 1. Conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Ubicar el archivo
Ajusta `RUTA` si tu carpeta de Drive se llama distinto. Si lo que subiste es un `.zip`, descomenta la línea de `unzip`.

In [ ]:
import os, glob

RUTA = '/content/drive/MyDrive/ML_LendingClub'   # <- cambia esto si tu carpeta se llama distinto

print('Contenido de la carpeta:')
for f in sorted(os.listdir(RUTA)):
    tam = os.path.getsize(os.path.join(RUTA, f)) / 1e6
    print(f'  {f}  ({tam:,.0f} MB)')

# Si subiste el .zip de Kaggle, descomenta la siguiente línea (solo la primera vez):
# !unzip -n "{RUTA}"/*.zip -d "{RUTA}"

In [ ]:
archivos = sorted(glob.glob(RUTA + '/*accepted*'))
print('Archivos candidatos (aceptados):', archivos)

## 3. Declarar el hardware
El enunciado pide declarar núcleos y memoria para que la comparación de tiempos sea interpretable. Lo dejamos registrado desde ya.

In [ ]:
import os, psutil
print('Núcleos (CPU):', os.cpu_count())
print('RAM total (GB):', round(psutil.virtual_memory().total / 1e9, 1))

## 4. Cargar el CSV completo
`low_memory=False` hace que pandas lea las columnas con tipos consistentes (evita advertencias de tipos mixtos). Medimos el tiempo de carga porque el enunciado permite compararlo con Spark.

In [ ]:
import pandas as pd, time

t0 = time.time()
df = pd.read_csv(archivos[0], low_memory=False)
t_carga = time.time() - t0

print(f'Tiempo de carga con pandas: {t_carga:,.1f} s')
print('Dimensión (filas, columnas):', df.shape)

## 5. Revisar `loan_status`
Esta columna es el origen del target. El enunciado define `default = 1` si `loan_status == "Charged Off"` y `0` en otro caso. Hay que ver **qué otras categorías existen** (por ejemplo, préstamos aún vigentes) porque cambian lo que significa el 0.

In [ ]:
conteo = df['loan_status'].value_counts(dropna=False)
print(conteo)
print()
print('Total filas:', len(df))
print('Fully Paid + Charged Off:', int(conteo.get('Fully Paid', 0) + conteo.get('Charged Off', 0)))

In [ ]:
print('Columna id presente:', 'id' in df.columns)
print('Primeras 5 columnas:', list(df.columns[:5]))
df[['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'purpose', 'home_ownership', 'addr_state', 'loan_status']].head(5)

## 👉 Qué copiar y pegarme en el chat
1. El nombre del archivo y su tamaño (sección 2).
2. Núcleos y RAM de este Colab (sección 3).
3. Tiempo de carga y dimensión (sección 4).
4. La tabla completa de `loan_status` (sección 5).

Con eso decidimos juntas, antes de seguir, cómo definir el 0 del target y qué conjunto es el "dataset completo" del enunciado.